# Himawari-9 month-scale AdvectNet + optional RAFT fine-tuning

This notebook trains on one month of Himawari-9 10-minute B13 thermal infrared data.

Pipeline:

1. Download Himawari-9 B13 frames from public NOAA S3.
2. Convert to brightness temperature and normalize exactly like the earlier notebook.
3. Build resumable daily patch shards containing `(X0, X3, Y1, Y2)`:
   - `X0 = t`
   - `Y1 = actual t+10`
   - `Y2 = actual t+20`
   - `X3 = t+30`
4. Train AdvectNet in stages:
   - Stage 1: frozen RAFT, train U-Net.
   - Stage 2: unfreeze selected RAFT blocks and fine-tune with small LR.
5. Save checkpoints containing both `raft` and `unet` weights.

This is designed to survive Kaggle session limits: shard creation and checkpoints are resumable.

## 1. Setup

In [ ]:
!pip install -q s3fs satpy scikit-image imageio
!pip uninstall -y -q cartopy

import os, bz2, shutil, datetime as dt, math, random, gc, json
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import s3fs
from satpy import Scene
from skimage.metrics import structural_similarity as ssim_fn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Configuration

In [ ]:
# ---------------- Data period ----------------
TRAIN_START_DATE = dt.date(2024, 6, 1)
NUM_DAYS = 30
VAL_DAYS = 3                    # last VAL_DAYS become validation shards

# ---------------- Himawari source ----------------
BUCKET = "noaa-himawari9"
PRODUCT = "AHI-L1b-FLDK"
BAND = "B13"
STEP_MIN = 10
FRAMES_PER_DAY = 24 * 60 // STEP_MIN
INDIA_BBOX = (82.0, 8.0, 100.0, 26.0)   # min_lon, min_lat, max_lon, max_lat
BT_MIN, BT_MAX = 180.0, 310.0

# ---------------- Patch sampling ----------------
PATCH = 256
STRIDE = 192
MAX_INVALID = 0.20
PATCHES_PER_WINDOW = 2          # month size control. 2 gives about 8.5k samples per 30 days if every window valid.
RNG_SEED = 0

# ---------------- Storage ----------------
WORK = Path("/kaggle/working")
SHARD_DIR = WORK / "hima_month_shards"
CKPT_DIR = WORK / "hima_month_checkpoints"
TMP = WORK / "hima_tmp"
SHARD_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Training ----------------
DEV = "cuda" if torch.cuda.is_available() else "cpu"
BS = 2                          # use 1 if Kaggle T4 OOMs during RAFT fine-tuning
NUM_WORKERS = 0                 # keep 0 because shard cache lives in the Dataset object
STAGE1_EPOCHS = 8               # frozen RAFT, U-Net only
STAGE2_EPOCHS = 4               # RAFT partial fine-tune; increase if stable
UNET_LR = 5e-4
RAFT_LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
AMP = True

# Loss weights.
LAMBDA_CHARB = 1.0
LAMBDA_GRAD = 0.5
LAMBDA_ADV = 0.05               # keep weaker during RAFT fine-tuning
LAMBDA_SRC = 0.02
LAMBDA_SMOOTH = 0.01

print("period:", TRAIN_START_DATE, "days:", NUM_DAYS)
print("shards:", SHARD_DIR)
print("checkpoints:", CKPT_DIR)

## 3. Himawari frame loader

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

def load_himawari_frame(t):
    """Return normalized B13 brightness-temperature crop and validity mask for one timestamp."""
    prefix = f"{BUCKET}/{PRODUCT}/{t:%Y/%m/%d/%H%M}/"
    try:
        files = sorted(f for f in fs.ls(prefix) if f"_{BAND}_" in f)
    except Exception as e:
        print("list failed", t, e)
        return None, None

    if len(files) < 10:
        print("not enough segments", t, len(files))
        return None, None

    TMP.mkdir(parents=True, exist_ok=True)
    paths = []
    try:
        for key in files:
            out = TMP / Path(key).name[:-4]
            with fs.open(key, "rb") as src:
                out.write_bytes(bz2.decompress(src.read()))
            paths.append(str(out))

        scn = Scene(reader="ahi_hsd", filenames=paths)
        scn.load([BAND], calibration="brightness_temperature")
        bt = np.asarray(scn.crop(ll_bbox=INDIA_BBOX)[BAND].values, dtype=np.float32)
    except Exception as e:
        print("read failed", t, e)
        return None, None
    finally:
        shutil.rmtree(TMP, ignore_errors=True)

    valid = np.isfinite(bt)
    norm = (np.clip(bt, BT_MIN, BT_MAX) - BT_MIN) / (BT_MAX - BT_MIN)
    norm = np.where(valid, norm, 0.0).astype(np.float32)
    return norm, valid.astype(bool)

def day_times(day):
    start = dt.datetime.combine(day, dt.time(0, 0))
    return [start + dt.timedelta(minutes=STEP_MIN * k) for k in range(FRAMES_PER_DAY)]

## 4. Build resumable daily patch shards

In [ ]:
def valid_patch_starts(h, w):
    ys = list(range(0, h - PATCH + 1, STRIDE))
    xs = list(range(0, w - PATCH + 1, STRIDE))
    return ys, xs

def sample_patches_for_window(quad, masks, rng):
    h, w = quad[0].shape
    ys, xs = valid_patch_starts(h, w)
    candidates = []
    for y in ys:
        for x in xs:
            sl = (slice(y, y + PATCH), slice(x, x + PATCH))
            vm = masks[0][sl] & masks[1][sl] & masks[2][sl] & masks[3][sl]
            if 1.0 - vm.mean() <= MAX_INVALID:
                candidates.append((y, x))
    if not candidates:
        return []
    rng.shuffle(candidates)
    return candidates[:min(PATCHES_PER_WINDOW, len(candidates))]

def build_day_shard(day, overwrite=False):
    shard_path = SHARD_DIR / f"hima_{day:%Y%m%d}.npz"
    if shard_path.exists() and not overwrite:
        print("exists, skip", shard_path.name)
        return shard_path

    rng = random.Random(RNG_SEED + int(day.strftime("%Y%m%d")))
    frames, masks, times = [], [], []
    target_shape = None

    print("\n=== building", day, "===")
    for k, t in enumerate(day_times(day)):
        f, m = load_himawari_frame(t)
        if f is None:
            frames.append(None); masks.append(None); times.append(t)
            print(f"[{k+1:03d}/{FRAMES_PER_DAY}] {t:%H:%M} missing")
            continue
        if target_shape is None:
            target_shape = f.shape
        h, w = target_shape
        if f.shape[0] < h or f.shape[1] < w:
            frames.append(None); masks.append(None); times.append(t)
            print(f"[{k+1:03d}/{FRAMES_PER_DAY}] {t:%H:%M} smaller shape skip")
            continue
        frames.append(f[:h, :w]); masks.append(m[:h, :w]); times.append(t)
        print(f"[{k+1:03d}/{FRAMES_PER_DAY}] {t:%H:%M} ok")

    if target_shape is None:
        print("no frames for day", day)
        return None

    X0, X3, Y1, Y2, t0s = [], [], [], [], []
    for i in range(len(frames) - 3):
        quad = frames[i:i+4]
        mq = masks[i:i+4]
        if any(x is None for x in quad):
            continue
        coords = sample_patches_for_window(quad, mq, rng)
        for y, x in coords:
            sl = (slice(y, y + PATCH), slice(x, x + PATCH))
            X0.append(quad[0][sl]); Y1.append(quad[1][sl]); Y2.append(quad[2][sl]); X3.append(quad[3][sl])
            t0s.append(times[i].strftime("%Y-%m-%dT%H:%M:%S"))

    if len(X0) == 0:
        print("no valid patches for day", day)
        return None

    np.savez_compressed(
        shard_path,
        X0=np.asarray(X0, np.float16),
        X3=np.asarray(X3, np.float16),
        Y1=np.asarray(Y1, np.float16),
        Y2=np.asarray(Y2, np.float16),
        t0=np.asarray(t0s),
        bt_min=np.float32(BT_MIN),
        bt_max=np.float32(BT_MAX),
        patch=np.int32(PATCH),
        bbox=np.asarray(INDIA_BBOX, np.float32),
    )
    print("saved", shard_path, "samples", len(X0), "size MB", shard_path.stat().st_size / 1e6)
    del frames, masks, X0, X3, Y1, Y2
    gc.collect()
    return shard_path

# Build all daily shards. Safe to rerun; existing shards are skipped.
days = [TRAIN_START_DATE + dt.timedelta(days=i) for i in range(NUM_DAYS)]
for day in days:
    build_day_shard(day, overwrite=False)

shards = sorted(SHARD_DIR.glob("hima_*.npz"))
print("\nshards built:", len(shards))
for p in shards[:5]: print(" ", p.name)
print("...")
for p in shards[-5:]: print(" ", p.name)

## 5. Dataset over daily shards

In [ ]:
class ShardPatchDataset(Dataset):
    def __init__(self, shard_paths, cache_size=2):
        self.shard_paths = list(shard_paths)
        self.cache_size = cache_size
        self.cache = OrderedDict()
        self.items = []
        self.lengths = []
        for si, p in enumerate(self.shard_paths):
            with np.load(p, allow_pickle=True) as d:
                n = len(d["X0"])
            self.lengths.append(n)
            for j in range(n):
                self.items.append((si, j))
        print("dataset shards", len(self.shard_paths), "samples", len(self.items))

    def __len__(self):
        return len(self.items)

    def _load_shard(self, si):
        if si in self.cache:
            self.cache.move_to_end(si)
            return self.cache[si]
        d = np.load(self.shard_paths[si], allow_pickle=True)
        pack = {k: d[k].astype(np.float32) if k in ["X0", "X3", "Y1", "Y2"] else d[k] for k in d.files}
        d.close()
        self.cache[si] = pack
        if len(self.cache) > self.cache_size:
            self.cache.popitem(last=False)
        return pack

    def __getitem__(self, idx):
        si, j = self.items[idx]
        d = self._load_shard(si)
        return d["X0"][j], d["X3"][j], d["Y1"][j], d["Y2"][j]

shards = sorted(SHARD_DIR.glob("hima_*.npz"))
assert len(shards) >= VAL_DAYS + 1, "Need more shards. Build at least VAL_DAYS+1 days."
train_shards = shards[:-VAL_DAYS]
val_shards = shards[-VAL_DAYS:]
print("train shards", len(train_shards), train_shards[0].name, "->", train_shards[-1].name)
print("val shards", len(val_shards), val_shards[0].name, "->", val_shards[-1].name)

train_ds = ShardPatchDataset(train_shards, cache_size=2)
val_ds = ShardPatchDataset(val_shards, cache_size=2)
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 6. Model definitions

In [ ]:
def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())

class UNet(nn.Module):
    def __init__(self, ic=8, b=32):
        super().__init__()
        self.e1 = nn.Sequential(cbr(ic, b), cbr(b, b))
        self.e2 = nn.Sequential(cbr(b, 2*b), cbr(2*b, 2*b))
        self.e3 = nn.Sequential(cbr(2*b, 4*b), cbr(4*b, 4*b))
        self.pool = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(cbr(4*b+2*b, 2*b), cbr(2*b, 2*b))
        self.d1 = nn.Sequential(cbr(2*b+b, b), cbr(b, b))
        self.out = nn.Conv2d(b, 2, 3, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        up = lambda z: F.interpolate(z, scale_factor=2, mode="bilinear", align_corners=False)
        d2 = self.d2(torch.cat([up(e3), e2], 1))
        d1 = self.d1(torch.cat([up(d2), e1], 1))
        o = self.out(d1)
        return torch.sigmoid(o[:, 0:1]), torch.tanh(o[:, 1:2])

raft = raft_small(weights=Raft_Small_Weights.DEFAULT).to(DEV)
unet = UNet().to(DEV)
print("models loaded")

## 7. Flow, warp, losses

In [ ]:
def set_raft_trainable(mode):
    for p in raft.parameters():
        p.requires_grad_(False)
    if mode == "frozen":
        raft.eval()
        return []
    trainable = []
    # Conservative partial fine-tune: update/recurrent motion machinery first.
    keys = ["update_block", "motion_encoder", "recurrent_block", "flow_head", "mask_predictor"]
    for name, p in raft.named_parameters():
        if any(k in name for k in keys):
            p.requires_grad_(True)
            trainable.append(p)
    # Fallback if torchvision naming changes.
    if not trainable:
        for name, p in raft.named_parameters():
            if "update" in name or "flow" in name:
                p.requires_grad_(True)
                trainable.append(p)
    raft.train()
    print("trainable RAFT tensors:", len(trainable))
    return trainable


def raft_flow(a, b):
    a3 = (a * 2 - 1).repeat(1, 3, 1, 1)
    b3 = (b * 2 - 1).repeat(1, 3, 1, 1)
    return raft(a3, b3)[-1]


def backwarp(img, flow):
    B, C, h, w = img.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=img.device), torch.arange(w, device=img.device), indexing="ij")
    grid = torch.stack((xx, yy), 0).float()[None].repeat(B, 1, 1, 1) + flow
    gx = 2 * grid[:, 0] / max(w - 1, 1) - 1
    gy = 2 * grid[:, 1] / max(h - 1, 1) - 1
    return F.grid_sample(img, torch.stack((gx, gy), -1), mode="bilinear", padding_mode="border", align_corners=True)


def inter_flows(F03, F30, a):
    Ft0 = -(1-a) * a * F03 + a * a * F30
    Ft3 = (1-a)**2 * F03 - a * (1-a) * F30
    return Ft0, Ft3


def forward_model(X0b, X3b, a, train_raft=False):
    if train_raft:
        F03 = raft_flow(X0b, X3b)
        F30 = raft_flow(X3b, X0b)
    else:
        with torch.no_grad():
            F03 = raft_flow(X0b, X3b)
            F30 = raft_flow(X3b, X0b)
    Ft0, Ft3 = inter_flows(F03, F30, a)
    w0 = backwarp(X0b, Ft0)
    w3 = backwarp(X3b, Ft3)
    blend = (1-a) * X0b + a * X3b
    ach = a.expand(-1, 1, X0b.shape[-2], X0b.shape[-1])
    mask, res = unet(torch.cat([w0, w3, Ft0, Ft3, blend, ach], 1))
    pred = mask * w0 + (1-mask) * w3 + res
    return pred, res, Ft0, Ft3, F03, F30


def charb(x, y, eps=1e-3):
    return torch.sqrt((x-y)**2 + eps**2).mean()


def gloss(x, y):
    return (((x[..., 1:] - x[..., :-1]).abs() - (y[..., 1:] - y[..., :-1]).abs()).abs().mean()
          + ((x[..., 1:, :] - x[..., :-1, :]).abs() - (y[..., 1:, :] - y[..., :-1, :]).abs()).abs().mean())


def spatial_gradient(x):
    dx = F.pad(x[..., 1:] - x[..., :-1], (0, 1))
    dy = F.pad(x[..., 1:, :] - x[..., :-1, :], (0, 0, 0, 1))
    return dx, dy


def advection_loss(pred, x0, Ft0):
    dpx, dpy = spatial_gradient(pred)
    adv = (pred - x0) + Ft0[:, 0:1] * dpx + Ft0[:, 1:2] * dpy
    return charb(adv, torch.zeros_like(adv))


def flow_smooth_loss(flow):
    dx = flow[..., 1:] - flow[..., :-1]
    dy = flow[..., 1:, :] - flow[..., :-1, :]
    return dx.abs().mean() + dy.abs().mean()


def total_loss(pred, y, res, Ft0, Ft3, x0):
    l_rec = charb(pred, y)
    l_grad = gloss(pred, y)
    l_adv = advection_loss(pred, x0, Ft0)
    l_src = res.abs().mean()
    l_sm = flow_smooth_loss(Ft0) + flow_smooth_loss(Ft3)
    loss = (LAMBDA_CHARB*l_rec + LAMBDA_GRAD*l_grad + LAMBDA_ADV*l_adv
            + LAMBDA_SRC*l_src + LAMBDA_SMOOTH*l_sm)
    return loss, {"rec": l_rec.detach(), "grad": l_grad.detach(), "adv": l_adv.detach(), "src": l_src.detach(), "smooth": l_sm.detach()}

## 8. Training and validation loops

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(AMP and DEV == "cuda"))

def batch_to_device(batch):
    X0b, X3b, Y1b, Y2b = batch
    X0b = X0b[:, None].to(DEV, non_blocking=True).float()
    X3b = X3b[:, None].to(DEV, non_blocking=True).float()
    Y1b = Y1b[:, None].to(DEV, non_blocking=True).float()
    Y2b = Y2b[:, None].to(DEV, non_blocking=True).float()
    return X0b, X3b, Y1b, Y2b


def train_epoch(loader, opt, train_raft=False, max_batches=None):
    unet.train()
    raft.train() if train_raft else raft.eval()
    sums = {"loss": 0.0, "rec": 0.0, "grad": 0.0, "adv": 0.0, "src": 0.0, "smooth": 0.0}
    n = 0
    for bi, batch in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        X0b, X3b, Y1b, Y2b = batch_to_device(batch)
        choose = torch.rand(X0b.shape[0], device=DEV) < 0.5
        y = torch.where(choose[:, None, None, None], Y1b, Y2b)
        avec = torch.where(choose, torch.full_like(choose.float(), 1/3), torch.full_like(choose.float(), 2/3))
        a = avec.view(-1, 1, 1, 1)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(AMP and DEV == "cuda")):
            pred, res, Ft0, Ft3, *_ = forward_model(X0b, X3b, a, train_raft=train_raft)
            loss, parts = total_loss(pred, y, res, Ft0, Ft3, X0b)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), GRAD_CLIP)
        if train_raft:
            torch.nn.utils.clip_grad_norm_([p for p in raft.parameters() if p.requires_grad], GRAD_CLIP)
        scaler.step(opt)
        scaler.update()

        sums["loss"] += float(loss.detach().cpu())
        for k, v in parts.items():
            sums[k] += float(v.cpu())
        n += 1
        if (bi + 1) % 100 == 0:
            print(f"batch {bi+1} loss {sums['loss']/n:.4f} rec {sums['rec']/n:.4f}")
    return {k: v / max(n, 1) for k, v in sums.items()}


@torch.no_grad()
def validate(loader, max_batches=None):
    unet.eval(); raft.eval()
    mse10, mse20, bmse10, bmse20 = [], [], [], []
    for bi, batch in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        X0b, X3b, Y1b, Y2b = batch_to_device(batch)
        for alpha, target, arr, barr in [(1/3, Y1b, mse10, bmse10), (2/3, Y2b, mse20, bmse20)]:
            a = torch.full((X0b.shape[0], 1, 1, 1), float(alpha), device=DEV)
            pred, *_ = forward_model(X0b, X3b, a, train_raft=False)
            pred = pred.clamp(0, 1)
            base = ((1-alpha) * X0b + alpha * X3b).clamp(0, 1)
            arr.extend(((pred - target) ** 2).mean(dim=(1,2,3)).detach().cpu().numpy().tolist())
            barr.extend(((base - target) ** 2).mean(dim=(1,2,3)).detach().cpu().numpy().tolist())
    return {
        "mse10": float(np.mean(mse10)),
        "mse20": float(np.mean(mse20)),
        "mse": float(np.mean(mse10 + mse20)),
        "baseline_mse10": float(np.mean(bmse10)),
        "baseline_mse20": float(np.mean(bmse20)),
        "baseline_mse": float(np.mean(bmse10 + bmse20)),
    }


def save_ckpt(name, stage, epoch, opt=None, metrics=None):
    path = CKPT_DIR / name
    obj = {
        "raft": raft.state_dict(),
        "unet": unet.state_dict(),
        "stage": stage,
        "epoch": epoch,
        "metrics": metrics or {},
        "config": {
            "bt_min": BT_MIN, "bt_max": BT_MAX, "patch": PATCH, "bbox": INDIA_BBOX,
            "train_start_date": str(TRAIN_START_DATE), "num_days": NUM_DAYS,
        },
    }
    if opt is not None:
        obj["optimizer"] = opt.state_dict()
    torch.save(obj, path)
    print("saved", path)
    return path

## 9. Stage 1: frozen RAFT, train U-Net

In [ ]:
set_raft_trainable("frozen")
opt1 = torch.optim.AdamW(unet.parameters(), lr=UNET_LR, weight_decay=WEIGHT_DECAY)
history = []

for ep in range(1, STAGE1_EPOCHS + 1):
    tr = train_epoch(train_loader, opt1, train_raft=False)
    va = validate(val_loader)
    row = {"stage": "stage1_frozen_raft", "epoch": ep, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    history.append(row)
    print(row)
    save_ckpt(f"advectnet_stage1_ep{ep:03d}.pth", "stage1_frozen_raft", ep, opt1, va)
    pd.DataFrame(history).to_csv(CKPT_DIR / "training_history.csv", index=False)

print("stage 1 done")

## 10. Stage 2: partial RAFT fine-tuning

In [ ]:
raft_params = set_raft_trainable("partial")
opt2 = torch.optim.AdamW([
    {"params": unet.parameters(), "lr": UNET_LR * 0.2},
    {"params": raft_params, "lr": RAFT_LR},
], weight_decay=WEIGHT_DECAY)

for ep in range(1, STAGE2_EPOCHS + 1):
    tr = train_epoch(train_loader, opt2, train_raft=True)
    va = validate(val_loader)
    row = {"stage": "stage2_partial_raft", "epoch": ep, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    history.append(row)
    print(row)
    save_ckpt(f"advectnet_raft_finetuned_ep{ep:03d}.pth", "stage2_partial_raft", ep, opt2, va)
    pd.DataFrame(history).to_csv(CKPT_DIR / "training_history.csv", index=False)

final_path = save_ckpt("advectnet_raft_finetuned_himawari_month.pth", "stage2_partial_raft", STAGE2_EPOCHS, opt2, va)
print("FINAL:", final_path)

## 11. Validation plots and examples

In [ ]:
hist_path = CKPT_DIR / "training_history.csv"
hist = pd.read_csv(hist_path)
display(hist.tail())

plt.figure(figsize=(8,4))
plt.plot(hist.index, hist["val_mse"], marker="o", label="model val mse")
plt.plot(hist.index, hist["val_baseline_mse"], marker="o", label="linear baseline val mse")
plt.xlabel("checkpoint index")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.savefig(CKPT_DIR / "val_mse_curve.png", dpi=140, bbox_inches="tight")
plt.show()

# Visualize one validation batch.
batch = next(iter(val_loader))
X0b, X3b, Y1b, Y2b = batch_to_device(batch)
with torch.no_grad():
    a1 = torch.full((X0b.shape[0],1,1,1), 1/3, device=DEV)
    a2 = torch.full((X0b.shape[0],1,1,1), 2/3, device=DEV)
    P1, *_ = forward_model(X0b, X3b, a1, train_raft=False)
    P2, *_ = forward_model(X0b, X3b, a2, train_raft=False)

nshow = min(3, X0b.shape[0])
fig, ax = plt.subplots(nshow, 8, figsize=(22, 3*nshow))
if nshow == 1:
    ax = ax[None, :]
cols = ["t", "t+30", "gen t+10", "actual t+10", "|err| t+10", "gen t+20", "actual t+20", "|err| t+20"]
for r in range(nshow):
    imgs = [X0b[r,0], X3b[r,0], P1[r,0].clamp(0,1), Y1b[r,0], (P1[r,0]-Y1b[r,0]).abs(), P2[r,0].clamp(0,1), Y2b[r,0], (P2[r,0]-Y2b[r,0]).abs()]
    imgs = [x.detach().cpu().numpy() for x in imgs]
    for c, im in enumerate(imgs):
        vmax = 1 if c not in [4,7] else max(0.05, float(np.percentile(im, 99)))
        ax[r,c].imshow(im, cmap="gray_r", vmin=0, vmax=vmax)
        ax[r,c].axis("off")
        if r == 0:
            ax[r,c].set_title(cols[c], fontsize=10)
plt.tight_layout()
plt.savefig(CKPT_DIR / "validation_examples.png", dpi=140, bbox_inches="tight")
plt.show()

## Notes for INSAT TIR1 testing

The final checkpoint contains both models:

```python
ckpt = torch.load("advectnet_raft_finetuned_himawari_month.pth", map_location=dev)
raft.load_state_dict(ckpt["raft"])
unet.load_state_dict(ckpt["unet"])
```

Use this checkpoint in the INSAT-3DR TIR1 MOSDAC test notebook. Because it contains fine-tuned RAFT weights, the test notebook must load `ckpt["raft"]` as well as `ckpt["unet"]`.